# Interferometer trajectory plotting

This notebook demonstrates `aispy.trajectory`: loading snapshot data from an
ais++ trajectory run and reconstructing smooth spacetime diagrams.

### How it works

Setting `printtrajectory 1` in the `.aisi` file causes ais++ to record
the position and velocity of every wavepacket at each pulse boundary
(before and after each pulse, plus the initial state and detection time).
Between snapshots the motion is analytic free-flight under gravity:
$$z(t) = z_0 + v_z(t_0)\,(t-t_0) - \tfrac{1}{2}g(t-t_0)^2$$
so the trajectory can be reconstructed at arbitrary resolution without
approximation for `linear_pot`.

### Intended use

`printtrajectory 1` is designed for **single-atom** (`natoms 1`) runs to
visualise the interferometer geometry.  Running with many atoms produces
very large files.

### Setup

```bash
pip install aispy   # or: pip install -e /path/to/aispy

# Generate the trajectory (from the aispp/trajectory-plots branch):
cd /path/to/aispp/examples
ais++ -i input-files/TRAJ_MZ_N1.aisi -o output-files/TRAJ_MZ_N1.h5
# This produces output-files/TRAJ_MZ_N1_TRAJ.h5
```
A pre-generated `TRAJ_MZ_N1_TRAJ.h5` is included in the aispp repo so you
can run this notebook without building ais++.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, '..')

from aispy.trajectory import load_trajectory, reconstruct_trajectories, \
                              plot_trajectory, plot_arm_separation

try:
    plt.style.use('seaborn-v0_8-ticks')
except OSError:
    plt.style.use('seaborn-ticks')
plt.rcParams.update({
    'font.family': 'serif', 'font.size': 9,
    'axes.labelsize': 9, 'savefig.dpi': 150,
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'legend.frameon': False,
})

AISPP_EXAMPLES = '../../aispp/examples'   # adjust if needed

# n=1001 LMT trajectory (generate with ais++ first — see Setup above)
TRAJ_N1001 = os.path.join(AISPP_EXAMPLES, 'output-files', 'TRAJ_MZ_N1001_TRAJ.h5')
# n=1 simple MZ (pre-generated, ships with the repo)
TRAJ_N1    = os.path.join(AISPP_EXAMPLES, 'output-files', 'TRAJ_MZ_N1_TRAJ.h5')

if not os.path.exists(TRAJ_N1001):
    print('⚠  TRAJ_MZ_N1001_TRAJ.h5 not found.')
    print('   Generate it with:')
    print(f'     cd {AISPP_EXAMPLES}')
    print('     ais++ -i input-files/TRAJ_MZ_N1001.aisi \\')
    print('           -o output-files/TRAJ_MZ_N1001.h5')
    print('   Then re-run this cell.')
else:
    print('n=1001 trajectory file found ✓')

## 1. Inspect the raw snapshot data

In [ ]:
traj = load_trajectory(TRAJ_N1001)

snap_times  = traj['snapshot_times']
snap_labels = traj['snapshot_labels']
positions   = traj['positions']
snap_idx    = np.array(traj['snapshot_idx'])

print(f"LMT order n=1001")
print(f"Snapshots : {len(snap_times)}  (one per pulse boundary)")
print(f"Records   : {len(traj['paths'])}")
print(f"z range   : {positions[:,2].min():.2f} → {positions[:,2].max():.2f} m")

# Find max arm separation
max_sep, max_t = 0, 0
for si in range(len(snap_times)):
    rec = snap_idx == si
    if rec.sum() < 2: continue
    zs = positions[rec, 2]
    sep = abs(zs.max() - zs.min())
    if sep > max_sep:
        max_sep, max_t = sep, snap_times[si]

print(f"\nMax arm separation : {max_sep:.3f} m  at t = {max_t:.4f} s")

# Expected: each recoil kick adds ħkz/m to the relative velocity
# For n=1001, net relative velocity = n × ħkz/m
hbar = 1.054571817e-34; kz_val = 9e6; m = 87 * 1.66054e-27
dv_rel = 1001 * hbar * kz_val / m
T_eff  = 2.225 - 1000 * (np.pi / (2e3 * np.pi) + 1e-7)  # account for LMT block duration
print(f"Expected (n·ħkz/m·T_eff = {dv_rel:.4f} m/s × {T_eff:.3f} s) : {dv_rel*T_eff:.3f} m")

# Compare with n=1
traj1 = load_trajectory(TRAJ_N1)
pos1 = traj1['positions']; si1 = np.array(traj1['snapshot_idx']); st1 = np.array(traj1['snapshot_times'])
max_sep1, max_t1 = 0, 0
for si in range(len(st1)):
    rec = si1 == si
    if rec.sum() < 2: continue
    zs = pos1[rec, 2]
    sep = abs(zs.max() - zs.min())
    if sep > max_sep1:
        max_sep1, max_t1 = sep, st1[si]
print(f"\nFor comparison, n=1: max separation = {max_sep1*100:.1f} cm")
print(f"Ratio n=1001/n=1 ≈ {max_sep/max_sep1:.0f}  (expected ≈ {1001:.0f})")

## 2. Spacetime diagram and transverse trajectory

The classic interferometer spacetime diagram: $z$ vs $t$ shows the
two arms separating after the first beam splitter, being redirected
by the mirror pulse, and recombining at the final beam splitter.
Grey bands mark the pulse durations (barely visible at this scale —
pulse duration $\tau \approx 0.25\,\text{ms} \ll T = 2.225\,\text{s}$).

In [ ]:
# n=1001: dense LMT snapshots cover the LMT blocks; interpolation only
# kicks in for the long free-flight intervals (dt > 5 ms threshold).
fig, (ax_z, ax_x) = plot_trajectory(traj, figsize=(9, 6), n_interp=200,
                                     show_pulses=False)   # too many pulses to shade

# Annotate arm separation
ax_z.annotate(f'arm separation\n$\\Delta z \\approx {max_sep:.1f}$ m',
              xy=(max_t, 0), xytext=(max_t * 0.6, 5),
              arrowprops=dict(arrowstyle='->', color='#555'),
              fontsize=8, ha='center')
ax_z.set_title(f'LMT Mach–Zehnder spacetime diagram ($n=1001$, $T=2.225$ s)', fontsize=9)
plt.savefig('trajectory_n1001_spacetime.png', bbox_inches='tight')
plt.show()

## 3. Arm separation vs time

The area enclosed by the two arms in the spacetime diagram is
proportional to $\hbar k_z g T^2$, the MZ gravitational phase.

In [ ]:
fig_sep, ax_sep = plot_arm_separation(traj, figsize=(7, 3.5))
ax_sep.axhline(max_sep * 100, color='gray', ls='--', lw=0.8, alpha=0.6,
               label=f'max = {max_sep:.2f} m')
ax_sep.set_title(f'Arm separation $|\\Delta z|(t)$ — LMT $n=1001$')
ax_sep.legend(fontsize=8)
plt.savefig('trajectory_n1001_arm_sep.png', bbox_inches='tight')
plt.show()

## 4. Reconstruct manually and inspect

Access the raw reconstructed data for custom analysis or plotting.

## 4. Side-by-side comparison: n=1 vs n=1001

Overlaying both trajectories on a shared time axis makes the LMT arm
separation obvious.

In [ ]:
_STATE_COLOR = {0: '#2ca02c', 1: '#d62728'}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=False,
                          gridspec_kw=dict(wspace=0.35))

for ax, (traj_i, title_i) in zip(axes, [
        (traj1,  r'$n=1$ (simple MZ)'),
        (traj,   r'$n=1001$ (LMT)')]):
    smooth_i = reconstruct_trajectories(traj_i, n_interp=200)
    seen = set()
    for path, d in smooth_i.items():
        col = _STATE_COLOR.get(d['state'], 'gray')
        lw  = max(0.5, 1.5 * d['amplitude'])
        label = f"state {d['state']}" if d['state'] not in seen else None
        ax.plot(d['t'], d['z'] * 100, color=col, lw=lw, alpha=0.85, label=label)
        seen.add(d['state'])
    ax.set_xlabel('$t$ [s]')
    ax.set_ylabel('$z$ [cm]')
    ax.set_title(title_i)
    ax.legend(fontsize=8)

plt.suptitle('Interferometer spacetime diagram: arm separation scales with LMT order', y=1.01)
plt.savefig('trajectory_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
smooth1001 = reconstruct_trajectories(traj,    atom_idx=0)
smooth1    = reconstruct_trajectories(traj1,   atom_idx=0)

print("n=1001 reconstructed paths:")
for path, d in smooth1001.items():
    print(f"  {path[:8]}...  state={d['state']}  amp={d['amplitude']:.3f}  "
          f"z=[{d['z'][0]*100:.1f}, {d['z'][-1]*100:.1f}] cm  ({len(d['t'])} pts)")

print("\nn=1 reconstructed paths:")
for path, d in smooth1.items():
    print(f"  {path:>6}  state={d['state']}  amp={d['amplitude']:.3f}  "
          f"z=[{d['z'][0]*100:.1f}, {d['z'][-1]*100:.1f}] cm  ({len(d['t'])} pts)")

## 5. Overlay multiple atoms — effect of initial conditions

Run ais++ with `natoms > 1` (e.g., a small psgrid with a few representative
atoms) and overlay their trajectories.  This shows how different initial
positions and velocities change the spacetime path.

```bash
# Example: 3×1 grid  (x0 = -100, 0, +100 µm;  vx0 = 0 fixed)
python build_psgrid_input.py --nx 3 --nvx 1 --stem TRAJ_3ATOMS
# then add  printtrajectory 1  to the generated .aisi and run ais++
```

In [ ]:
# Placeholder — uncomment and adjust path once you have a multi-atom trajectory file
# MULTI_TRAJ_FILE = '../../aispp/examples/output-files/TRAJ_3ATOMS_TRAJ.h5'
# if os.path.exists(MULTI_TRAJ_FILE):
#     traj_multi = load_trajectory(MULTI_TRAJ_FILE)
#     n_atoms = int(np.asarray(traj_multi['atom_indices']).max()) + 1
#     fig, (az, ax) = plt.subplots(2, 1, figsize=(8, 5.5), sharex=True)
#     colors = plt.cm.viridis(np.linspace(0, 1, n_atoms))
#     for ai, col in zip(range(n_atoms), colors):
#         smooth_i = reconstruct_trajectories(traj_multi, atom_idx=ai)
#         for path, d in smooth_i.items():
#             az.plot(d['t'], d['z']*100, color=col, lw=0.8, alpha=0.7)
#             ax.plot(d['t'], d['x']*1e3, color=col, lw=0.8, alpha=0.7)
#     az.set_ylabel('z [cm]'); ax.set_ylabel('x [mm]'); ax.set_xlabel('t [s]')
#     plt.tight_layout(); plt.show()
# else:
print("Multi-atom trajectory file not found — run ais++ with a psgrid + printtrajectory 1.")